# Pandas Series 综合案例练习

说明：本练习基于课件案例进行了改编与扩展，重点覆盖随机生成数据、布尔筛选、排序、差分、时间索引、收益率、重采样、滚动统计等 Series 场景。请直接在题目下方代码单元中完成。

In [119]:
import pandas as pd
import numpy as np

## 案例1：学生成绩分析

使用随机数创建一个包含 12 名学生数学成绩的 Series，分数范围 50 到 100，索引格式为 `学生1` 到 `学生12`。

要求：

1. 固定随机种子为 `42`。
2. 输出平均分、最高分、最低分。
3. 输出高于平均分的学生名单与人数。
4. 将成绩按从高到低排序。
5. 统计 60 分及以下、61 到 80 分、81 分及以上三个区间分别有多少人。

In [120]:
# 固定随机种子为42
np.random.seed(42)
# 创建包含12个学生数学成绩的series
# 注意如果randint(50, 101, 1)返回的是长度为1的数组
math_series = pd.Series(np.random.randint(50, 101, 12),
                        index=["学生"+ str(i) for i in range(1, 13, 1)])
# 输出平均分、最高分、最低分
print(f"平均分={math_series.mean():.2f}, 最高分={math_series.max()}, 最低分={math_series.min()}")
# 输出高于平均分的学生名单与人数
above_average = math_series.loc[math_series > math_series.mean()]
print(f"高出平均分的学生名单\n{above_average}, 人数:{above_average.count()}")
# 将成绩按从高到低排序
math_series = math_series.sort_values(ascending=False)
# 统计 60 分及以下、61 到 80 分、81 分及以上三个区间分别有多少人
print(f"60分及以下:{math_series.loc[math_series <= 60].count()}人")
print(f"61到80分:{math_series.loc[(math_series >= 61) & (math_series <= 80)].count()}人")
print(f"81分及以上:{math_series.loc[math_series >= 81].count()}人")


平均分=72.50, 最高分=92, 最低分=57
高出平均分的学生名单
学生1     88
学生2     78
学生4     92
学生7     88
学生12    73
dtype: int32, 人数:5
60分及以下:3人
61到80分:6人
81分及以上:3人


## 案例2：一周温度数据分析

已知某城市一周最高温度如下：

```python
temperatures = pd.Series(
    [29, 33, 31, 35, 30, 28, 34],
    index=['周一', '周二', '周三', '周四', '周五', '周六', '周日']
)
```

要求：

1. 找出温度严格大于 30 度的日期和天数。
2. 计算平均温度。
3. 按温度从高到低排序。
4. 使用 `diff()` 结合绝对值分析每天温度变化幅度。
5. 找出温差最大的两天对应的日期。
6. 判断周末平均温度是否高于工作日平均温度。

In [121]:
temperatures = pd.Series(
    [29, 33, 31, 35, 30, 28, 34],
    index=['周一', '周二', '周三', '周四', '周五', '周六', '周日']
)

# 找出温度严格大于 30 度的日期和天数
above_thirty_days = temperatures.loc[temperatures > 30]
print(f"温度严格大于 30 度的日期:{above_thirty_days.index.tolist()} 天数:{above_thirty_days.count()}")
# 计算平均温度
print(f"平均气温:{temperatures.mean()}")
# 按温度从高到低排序
print("温度从高到低排序: ", temperatures.sort_values(ascending=False))
# 使用 `diff()` 结合绝对值分析每天温度变化幅度
temp_changes = temperatures.diff().abs()
# 找出温差最大的两天对应的日期
print(f"温差最大的两天对应的日期:{temp_changes.sort_values(ascending=False).head(2).index.tolist()}")
# 判断周末平均温度是否高于工作日平均温度
weekday_aver = temperatures.loc['周一': '周五'].mean()
weekend_aver = temperatures.loc['周六': '周日'].mean()
if weekend_aver > weekday_aver:
    print("周末平均温度高于工作日平均温度")
elif weekend_aver < weekday_aver:
    print("周末平均温度小于工作日平均温度")
else:
    print("周末平均温度等于工作日平均温度")




温度严格大于 30 度的日期:['周二', '周三', '周四', '周日'] 天数:4
平均气温:31.428571428571427
温度从高到低排序:  周四    35
周日    34
周二    33
周三    31
周五    30
周一    29
周六    28
dtype: int64
温差最大的两天对应的日期:['周日', '周五']
周末平均温度小于工作日平均温度


## 案例3：股票价格与收益率分析

已知 10 个交易日收盘价：

```python
prices = pd.Series(
    [102.3, 103.5, 105.1, 104.8, 106.2, 107.0, 106.5, 108.1, 109.3, 110.2],
    index=pd.date_range('2023-01-01', periods=10)
)
```

要求：

1. 计算每日收益率。
2. 找出收益率最高和最低的日期。
3. 计算收益率波动率，即收益率标准差。
4. 统计上涨天数、下跌天数。
5. 输出涨幅最高的前 3 天。
6. 计算整个区间的累计涨跌幅。

In [122]:
# 创建10个交易日收盘价的Series
prices = pd.Series(
    [102.3, 103.5, 105.1, 104.8, 106.2, 107.0, 106.5, 108.1, 109.3, 110.2],
    index=pd.date_range('2023-01-01', periods=10)
)
# 计算每日收益率
price_profit_rates = prices.pct_change()
print(prices.pct_change())
# 收益率最高和最低的日期
print("收益率最高日期:", price_profit_rates.idxmax(),
      "\n收益率最低日期:", price_profit_rates.idxmin())
# 计算收益率波动率，即收益率标准差
print("收益率波动率: ", price_profit_rates.std())
# 上涨天数，下跌天数
print(f"上涨天数:{price_profit_rates.loc[price_profit_rates > 0].count()}天, "
      f"\n下跌天数:{price_profit_rates.loc[price_profit_rates < 0].count()}天")
# 输出涨幅最高的前 3 天
print(f"涨幅最高的前 3 天:{price_profit_rates.sort_values(ascending=False).head(3).index.tolist()}")
# 计算整个区间的累计涨跌幅（区间的收尾价格来算）
print(f"累计涨跌幅={prices.iloc[-1]/prices.iloc[0] - 1: .2f}")

2023-01-01         NaN
2023-01-02    0.011730
2023-01-03    0.015459
2023-01-04   -0.002854
2023-01-05    0.013359
2023-01-06    0.007533
2023-01-07   -0.004673
2023-01-08    0.015023
2023-01-09    0.011101
2023-01-10    0.008234
Freq: D, dtype: float64
收益率最高日期: 2023-01-03 00:00:00 
收益率最低日期: 2023-01-07 00:00:00
收益率波动率:  0.007373623845361105
上涨天数:7天, 
下跌天数:2天
涨幅最高的前 3 天:[Timestamp('2023-01-03 00:00:00'), Timestamp('2023-01-08 00:00:00'), Timestamp('2023-01-05 00:00:00')]
累计涨跌幅= 0.08


## 案例4：月度销量分析

已知某产品过去 12 个月销量如下：

```python
sales = pd.Series(
    [120, 135, 145, 160, 155, 170, 180, 175, 190, 200, 210, 220],
    index=pd.date_range('2022-01-01', periods=12, freq='MS')
)
```

要求：

1. 按季度重采样，计算每个季度平均销量。
2. 找出销量最高的月份和销量最低的月份。
3. 计算月环比增长率。
4. 找出连续增长超过 2 个月的月份。
5. 计算全年总销量与月均销量。
6. 统计下半年销量是否整体高于上半年。

In [123]:
# 创建销量series
sales = pd.Series(
    [120, 135, 145, 160, 155, 170, 180, 175, 190, 200, 210, 220],
    index=pd.date_range('2022-01-01', periods=12, freq='MS')
)
# 按季度重采样，计算每个季度平均销量
print(sales.resample('Q').mean())
# 找出销量最高的月份和销量最低的月份
print("销量最高的月份和销量最低的月份")
print(sales.idxmax(), sales.idxmin())
# 计算月环比增长率
s_pctchange = sales.pct_change()
print("月环比增长率")
print(s_pctchange)
# 找出连续增长超过 2 个月的月份(rolling为滑动窗口)
# 创建一个Series，判断当月是否涨
raise_sales = s_pctchange > 0
print("连续增长超过 2 个月的月份")
print(raise_sales[3 == raise_sales.rolling(3).sum()].index.tolist())
# 计算全年总销量与月均销量
print(f"全年总销量={sales.sum()}, 月均销量={sales.mean():.2f}")
# 统计下半年销量是否整体高于上半年
first_half_year_sales = sales[sales.index.month < 7].sum()
last_half_year_sales = sales[sales.index.month >= 7].sum()
print(f"下半年销量{'是' if last_half_year_sales > first_half_year_sales else '不是'}整体高于上半年")

2022-03-31    133.333333
2022-06-30    161.666667
2022-09-30    181.666667
2022-12-31    210.000000
Freq: Q-DEC, dtype: float64
销量最高的月份和销量最低的月份
2022-12-01 00:00:00 2022-01-01 00:00:00
月环比增长率
2022-01-01         NaN
2022-02-01    0.125000
2022-03-01    0.074074
2022-04-01    0.103448
2022-05-01   -0.031250
2022-06-01    0.096774
2022-07-01    0.058824
2022-08-01   -0.027778
2022-09-01    0.085714
2022-10-01    0.052632
2022-11-01    0.050000
2022-12-01    0.047619
Freq: MS, dtype: float64
连续增长超过 2 个月的月份
[Timestamp('2022-04-01 00:00:00'), Timestamp('2022-11-01 00:00:00'), Timestamp('2022-12-01 00:00:00')]
全年总销量=2060, 月均销量=171.67
下半年销量是整体高于上半年


## 案例5：小时级销售额分析

请生成某商店一天 24 小时销售额 Series：随机种子固定为 `42`，销售额使用 `np.random.randint(0, 100, 24)` 生成，索引使用 `pd.date_range('2025-01-01', periods=24, freq='H')`。

要求：

1. 计算当天总销售额。
2. 按天重采样，计算每日总销售额。
3. 计算营业时间（8:00 到 22:00，含 22:00）与非营业时间销售额占比。
4. 找出销售额最高的 3 个小时。
5. 计算每小时销售额的平均值、中位数和标准差。
6. 找出连续 3 个小时销售总额最高的时间段。

In [126]:
# 随机种子固定为42
np.random.seed(42)
# 生成销售额Series
sales_series = pd.Series(np.random.randint(0,100,24), index=pd.date_range('2025-01-01', periods=24, freq='H'))
# 计算当前总销售额
print(f"{sales_series.index.date[0]}总销售额为:{sales_series.sum()}")
# 按天重采样，计算每日总销售额
print(f"{sales_series.index.date[0]}总销售额为:{sales_series.resample('D').sum()[0]}")

# 计算营业时间（8:00 到 22:00，含 22:00）与非营业时间销售额占比(有两种方法)
# 方法一：利用between方法
# on_sales_time = sales_series.between_time("8:00", "22:00").sum()
# off_sales_time = sales_series.sum() - on_sales_time
# 方法二：利用索引
# mask: 布尔索引(用于判断是否是营业时间)
mask = (sales_series.index.hour >= 8) & (sales_series.index.hour <= 22)
on_sales_time = sales_series[mask].sum()
off_sales_time = sales_series[~mask].sum()
print(f"营业时间（8:00 到 22:00，含 22:00）与非营业时间销售额占比={on_sales_time / off_sales_time:.2f}")

# 销售额最高的 3 个小时
print("销售额最高的 3 个小时: ", sales_series.nlargest(3).index.tolist())

# 每小时销售额的平均值、中位数和标准差
print("每小时销售额的平均值")
print(sales_series.mean())
print("每小时销售额的中位数")
print(sales_series.median())
print("每小时销售额的标准差")
print(sales_series.std())

# 找出销售额最高的 3 个小时
# 实在不会，让GPT告诉我的
three_hour_sum = sales_series.rolling(3).sum()
max_end_time = three_hour_sum.idxmax()
print("连续 3 个小时销售总额最高的时间段")
print(sales_series.loc[(max_end_time-pd.Timedelta(hours=2)): max_end_time])


2025-01-01总销售额为:1205
2025-01-01总销售额为:1205
营业时间（8:00 到 22:00，含 22:00）与非营业时间销售额占比=1.43
销售额最高的 3 个小时:  [Timestamp('2025-01-01 11:00:00'), Timestamp('2025-01-01 01:00:00'), Timestamp('2025-01-01 10:00:00')]
每小时销售额的平均值
50.208333333333336
每小时销售额的中位数
55.5
每小时销售额的标准差
31.99997169382806
连续 3 个小时销售总额最高的时间段
2025-01-01 09:00:00    74
2025-01-01 10:00:00    87
2025-01-01 11:00:00    99
Freq: H, dtype: int32


## 案例6：缺失值与清洗练习

已知电商订单评分数据如下：

```python
ratings = pd.Series(
    [5, 4, np.nan, 3, 5, None, 2, 4, 5, 3],
    index=[f'订单{i}' for i in range(1, 11)],
    name='评分'
)
```

要求：

1. 检查哪些订单评分缺失。
2. 统计非缺失评分数量。
3. 计算现有评分的平均值、众数和各分值出现次数。
4. 将缺失值用平均分填充后，重新计算平均值。
5. 将评分按从高到低排序。
6. 统计唯一评分个数，并输出去重后的评分结果。

In [125]:
# 创建电商订单评分数据Series
ratings = pd.Series(
    [5, 4, np.nan, 3, 5, None, 2, 4, 5, 3],
    index=[f'订单{i}' for i in range(1, 11)],
    name='评分'
)
# 检查哪些订单评分缺失。
ratings[ratings.isna()]
# 统计非缺失评分数量。
ratings.count()
# 计算现有评分的平均值、众数和各分值出现次数。
ratings.mean(), *ratings.mode().values.tolist(), ratings.value_counts()
# 将缺失值用平均分填充后，重新计算平均值。
ratings = ratings.fillna(ratings.mean())
ratings.mean()
# 将评分按从高到低排序。
ratings.sort_values(ascending=False)
# 统计唯一评分个数，并输出去重后的评分结果。
ratings.nunique()
ratings.unique()

array([5.   , 4.   , 3.875, 3.   , 2.   ])